In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

# 1. 모델 초기화
model = init_chat_model(
    "google_genai:gemini-2.5-flash-lite",
    temperature=0
)

In [6]:
from typing import TypedDict, Annotated
import operator

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AnyMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

# 2. State 정의
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# 3. Node 정의
def llm_node(state: MessagesState):
    response = model.invoke(
        state["messages"]
    )
    return {"messages": [response]}

# 4. Graph 생성
graph_builder = StateGraph(MessagesState)

# 5. Graph에 Node 추가
graph_builder.add_node("llm", llm_node)

# 6. Edge 추가하여 Node 연결
graph_builder.add_edge(START, "llm")
graph_builder.add_edge("llm", END)

# 7. Graph를 실행 가능한 형태로 컴파일
checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

In [7]:
# 8. Graph 실행
config = {"configurable": {"thread_id": "conversation_1"}}

In [11]:
# get_state_history 테스트
human_message = HumanMessage(content="내 이름은 김일남이야.")
initial_message = {"messages": [human_message]}
result = graph.invoke(initial_message, config=config)

In [12]:
result

{'messages': [HumanMessage(content='내 이름은 김일남이야.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='안녕하세요, 김일남님. 만나서 반갑습니다!\n\n무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c6ea7-f9e1-7873-943a-3f448816f905-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 19, 'total_tokens': 28, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='내 이름은 김일남이야.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='네, 김일남님. 다시 한번 인사드립니다.\n\n혹시 제가 김일남님께 어떤 도움을 드릴 수 있을까요? 궁금한 점이 있으시거나, 하고 싶은 이야기가 있으시면 편하게 말씀해주세요.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c6eaa-c4fa-7160-80b7-e17f6eb8ffeb-0', tool_calls=[], invalid_tool_calls=[], usage_metad

In [22]:
state_history = graph.get_state_history(config=config)

In [23]:
state_history

<generator object Pregel.get_state_history at 0x0000021ED5FE7480>

In [15]:
for i, state in enumerate(state_history):
    print(f"--- State {i+1} ---")
    print(state)

--- State 1 ---
StateSnapshot(values={'messages': [HumanMessage(content='내 이름은 김일남이야.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 김일남님. 만나서 반갑습니다!\n\n무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c6ea7-f9e1-7873-943a-3f448816f905-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 19, 'total_tokens': 28, 'input_token_details': {'cache_read': 0}}), HumanMessage(content='내 이름은 김일남이야.', additional_kwargs={}, response_metadata={}), AIMessage(content='네, 김일남님. 다시 한번 인사드립니다.\n\n혹시 제가 김일남님께 어떤 도움을 드릴 수 있을까요? 궁금한 점이 있으시거나, 하고 싶은 이야기가 있으시면 편하게 말씀해주세요.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c6eaa-c4fa-7160-80b7-e17f6eb8ffeb-0', tool_calls=[], inv

In [21]:

print(list(state_history)[-1])

StateSnapshot(values={'messages': []}, next=('__start__',), config={'configurable': {'thread_id': 'conversation_1', 'checkpoint_ns': '', 'checkpoint_id': '1f10c749-511d-6911-bfff-87afece90c86'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-02-18T02:50:28.191054+00:00', parent_config=None, tasks=(PregelTask(id='6096ca89-b8ad-003b-6bbb-10b0293231c0', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'messages': [HumanMessage(content='내 이름은 김일남이야.', additional_kwargs={}, response_metadata={})]}),), interrupts=())


In [24]:
### 3. update_state 테스트 ###
human_message = HumanMessage(content="첨성대는 어느 도시에 있나요?")
initial_message = {"messages": [human_message]}
graph.invoke(initial_message, config=config)

graph.update_state(
    config=config,
    values={"messages": [HumanMessage(content="그 도시의 관광명소를 알려주세요.")]},
)

result = graph.invoke({"messages": []}, config=config)
print(result["messages"][-1].content)

경주에는 정말 많은 관광 명소가 있습니다! 김일남님께서 어떤 종류의 관광을 좋아하시는지에 따라 추천이 달라질 수 있지만, 일반적으로 많은 분들이 찾는 대표적인 명소들을 소개해 드릴게요.

**역사 유적 중심:**

*   **불국사:** 유네스코 세계문화유산으로 지정된 신라 시대의 대표적인 사찰입니다. 아름다운 건축물과 자연이 어우러져 있습니다.
*   **석굴암:** 불국사와 함께 유네스코 세계문화유산으로 지정된 곳으로, 정교한 석굴 건축과 석가모니 불상이 인상적입니다.
*   **대릉원 (천마총):** 신라 시대 왕릉들이 모여 있는 곳으로, 특히 천마총 내부를 관람하며 당시의 유물을 볼 수 있습니다.
*   **첨성대:** 우리나라에서 가장 오래된 천문 관측대로, 경주의 상징과도 같은 곳입니다.
*   **동궁과 월지 (안압지):** 신라 왕궁의 별궁터로, 연못에 비친 아름다운 야경이 유명합니다. 밤에 방문하면 더욱 멋집니다.
*   **경주 국립박물관:** 신라 시대의 찬란한 유물들을 체계적으로 전시하고 있어 신라의 역사를 깊이 이해할 수 있습니다.
*   **황룡사지:** 신라 시대 최대 규모의 사찰이었던 황룡사의 터를 볼 수 있습니다. 복원된 9층 목탑 모형도 있습니다.

**자연 및 휴식 공간:**

*   **보문관광단지:** 넓은 호수를 중심으로 호텔, 리조트, 놀이시설, 산책로 등이 잘 조성되어 있어 휴식을 취하기 좋습니다.
*   **토함산:** 불국사와 석굴암이 있는 산으로, 등산을 즐기거나 자연 속에서 힐링하기 좋습니다.
*   **주상절리 파도소리길:** 동해안의 멋진 해안 절경을 감상하며 산책할 수 있는 곳입니다.

**그 외:**

*   **교촌마을:** 최부잣집 고택과 전통 가옥들이 모여 있어 옛 정취를 느낄 수 있습니다. 떡갈비 등 맛있는 음식도 즐길 수 있습니다.
*   **월정교:** 복원된 아름다운 다리로, 야경이 특히 아름답습니다.

김일남님께서 혹시 특별히 관심 있는 분야 (예: 역사, 자연, 음식, 체험 등)가 있으신